In [2]:
import turtle
import easygui

def move_to(t, x, y):
    """Move the turtle without drawing."""
    t.penup()
    t.goto(x, y)
    t.setheading(0)
    t.pendown()

def draw_square(t, size):
    """Draw a square with side length `size`."""
    for _ in range(4):
        t.forward(size)
        t.left(90)

def draw_triangle(t, size):
    """Draw an equilateral triangle with side length `size`."""
    for _ in range(3):
        t.forward(size)
        t.left(120)

def draw_star(t, size):
    """Draw a 5-pointed star with edge length `size`."""
    for _ in range(5):
        t.forward(size)
        t.right(144)

def draw_concentric_circles(t, size, color):
    """Draw 4 concentric circles."""
    step = size // 4
    for i, radius in enumerate(range(size, 0, -step)):
        move_to(t, 0, -radius)
        t.fillcolor(color if i % 2 == 0 else "white")
        t.begin_fill()
        t.circle(radius)
        t.end_fill()

def get_user_inputs():
    shape = easygui.buttonbox(
        "Choose a shape to draw:",
        title="Pattern Studio",
        choices=["Square", "Equilateral Triangle", "Concentric Circles", "Custom Star"],
    )

    if shape is None:
        return None

    size = easygui.integerbox(
        "Enter the size / radius (50 to 250):",
        title="Size",
        default=100,
        lowerbound=50,
        upperbound=250,
    )

    if size is None:
        return None

    color = easygui.enterbox(
        "Enter a fill color (e.g. blue, red, teal):",
        title="Fill Color",
        default="blue",
    )

    if color is None:
        return None

    color = color.strip().lower() or "blue"
    return shape, size, color

def main():
    inputs = get_user_inputs()

    if inputs is None:
        easygui.msgbox("No drawing created. Goodbye!", "Cancelled")
        return

    shape, size, color = inputs

    screen = turtle.Screen()
    screen.setup(width=800, height=700)
    screen.title("Interactive Geometric Pattern Studio")

    t = turtle.Turtle()
    t.speed(3)
    t.pensize(2)

    try:
        t.color("black", color)
    except turtle.TurtleGraphicsError:
        easygui.msgbox(f"'{color}' is not a valid color. Using blue instead.", "Invalid Color")
        color = "blue"
        t.color("black", color)

    if shape == "Square":
        move_to(t, -size / 2, -size / 2)
        t.begin_fill()
        draw_square(t, size)
        t.end_fill()

    elif shape == "Equilateral Triangle":
        move_to(t, -size / 2, -size / 3)
        t.begin_fill()
        draw_triangle(t, size)
        t.end_fill()

    elif shape == "Custom Star":
        move_to(t, -size / 2, size / 6)
        t.begin_fill()
        draw_star(t, size)
        t.end_fill()

    elif shape == "Concentric Circles":
        draw_concentric_circles(t, size, color)

    t.hideturtle()
    easygui.msgbox("Drawing complete", "Done")
    turtle.bye()

if __name__ == "__main__":
    main()


In [4]:
import requests
from datetime import datetime

API_URL = "https://api.open-meteo.com/v1/forecast"
REPORT_FILE = "weather_report.txt"

# Short descriptions for WMO weather codes returned by Open-Meteo
WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    71: "Slight snow",
    73: "Moderate snow",
    75: "Heavy snow",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}

# Fetching data
def fetch_weather(lat, lon):
    """Send a GET request and return the parsed JSON, or None on failure."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "current_weather": "true",
        "timezone": "auto",
    }

    try:
        response = requests.get(API_URL, params=params, timeout=10)

        if response.status_code != 200:
            print(f"Error: server returned HTTP status {response.status_code}.")
            print(f"Details: {response.text[:200]}")
            return None

        return response.json()

    except requests.exceptions.Timeout:
        print("Error: the request timed out. Please try again.")

    except requests.exceptions.ConnectionError:
        print("Error: could not connect to the server. Check your internet connection.")

    except requests.exceptions.RequestException as e:
        print(f"Error: the request failed: {e}")

    except ValueError:
        print("Error: the server response was not valid JSON.")

    return None

# Parsing data
def parse_weather(data):
    """Extract the key metrics from the JSON response."""
    try:
        current = data["current_weather"]
        return {
            "temperature": current["temperature"],
            "windspeed": current["windspeed"],
            "weathercode": current["weathercode"],
            "timezone": data["timezone"],
        }

    except (KeyError, TypeError):
        print("Error: the response did not contain the expected weather data.")
        return None

# Formatting and saving
def build_summary(info, lat, lon):
    """Turn the extracted values into a human-readable text summary."""
    description = WEATHER_CODES.get(info["weathercode"], "Unknown conditions")
    timestamp = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

    lines = [
        "=" * 40,
        " WEATHER REPORT",
        "=" * 40,
        f"Generated at : {timestamp}",
        f"Location : latitude {lat}, longitude {lon}",
        f"Timezone : {info['timezone']}",
        f"Temperature : {info['temperature']} °C",
        f"Wind speed : {info['windspeed']} km/h",
        f"Weather code : {info['weathercode']} ({description})",
        "=" * 40,
    ]

    return "\n".join(lines) + "\n"

def save_report(text):
    """Save the report to a local log file."""
    try:
        with open(REPORT_FILE, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"Report saved to {REPORT_FILE}")

    except OSError as e:
        print(f"Error: could not write to {REPORT_FILE}: {e}")

# Input
def get_coordinates():
    """Ask for latitude and longitude, defaulting to Mumbai."""
    try:
        lat_text = input("Enter latitude (press Enter for Mumbai, 19.07): ").strip()
        lon_text = input("Enter longitude (press Enter for Mumbai, 72.88): ").strip()

        lat = float(lat_text) if lat_text else 19.07
        lon = float(lon_text) if lon_text else 72.88

    except ValueError:
        print("Invalid input. Using Mumbai's coordinates instead.")
        return 19.07, 72.88

    return lat, lon

# Main program
def main():
    lat, lon = get_coordinates()

    print("\nFetching weather data...")
    data = fetch_weather(lat, lon)

    if data is None:
        return

    info = parse_weather(data)

    if info is None:
        return

    summary = build_summary(info, lat, lon)

    print()
    print(summary)
    save_report(summary)

if __name__ == "__main__":
    main()



Fetching weather data...

 WEATHER REPORT
Generated at : 04-10-2026 19:09:38
Location : latitude 19.07, longitude 72.88
Timezone : Asia/Kolkata
Temperature : 30.5 °C
Wind speed : 9.9 km/h
Weather code : 0 (Clear sky)

Report saved to weather_report.txt
